In [ ]:
"""
    File: modello.ipynb
    Author: Andrea Macale
    Date: 2026-03-04

    Description: Notebook per la realizzazione del modello per il suggerimento ed analisi di follow-up clinici

"""

# Parte 0: Importazione delle librerie

In [2]:
%pip install -r requirements.txt -q

## Librerie principali

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

In [ ]:
import sys

# Si sposta nella directory principale dello Studio
os.chdir('/teamspace/studios/this_studio')

# Aggiunge la directory corrente al PATH per trovare 'src'
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

print(f"Directory attuale: {os.getcwd()}")

## Correlazione 

In [47]:
from sklearn.preprocessing import LabelEncoder
import scipy.stats
from scipy.stats import spearmanr

## VIF

In [48]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import statsmodels.api as sm

## Distribuzione dei dati

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Train

In [ ]:
import torch
import torchvision.transforms as T
import torch.nn as nn
from src.train.train import train_modello_visivo, train_modello_clinico
from src.models.DiagnosiIntegrata import DiagnosiIntegrata
from src.features.RXToraceDataset import RXToraceDataset
from tqdm.auto import tqdm

## Path

# Parte 1: Estrazione dei dati

## Aperta dei file del dataset

In [52]:
pos_dataset = os.path.join("data")

In [53]:
file = os.path.join(pos_dataset, "patients.csv")
pazienti = pd.read_csv(file)
pazienti['subject_id'] = pazienti['subject_id'].astype(str)
pazienti.head()

In [54]:
file = os.path.join(pos_dataset, "diagnoses_icd.csv")
tipi = {
    'subject_id': str, 
    'hadm_id': str, 
    'icd_code': str, 
    'icd_version': str
}
diagnosi = pd.read_csv(file, dtype=tipi)
diagnosi['icd_version'] = pd.to_numeric(diagnosi['icd_version'], errors='coerce').fillna(0).astype(int)
diagnosi = diagnosi.drop_duplicates()
diagnosi.head()

In [55]:
file = os.path.join(pos_dataset, "mimic-cxr-2.0.0-metadata.csv")
metadati = pd.read_csv(file)
metadati['dicom_id'] = metadati['dicom_id'].astype(str)
metadati['subject_id'] = metadati['subject_id'].astype(str)
metadati['study_id'] = metadati['study_id'].astype(str)
metadati.head()

In [ ]:
file = os.path.join(pos_dataset, "icustays.csv")
visite = pd.read_csv(file)
visite.head()

In [56]:
lista_immagini = []
ricerca = Path(os.path.join(pos_dataset, "MIMIC_SUPER_RES_24K"))
for file_path in ricerca.rglob('*.jpg'):
    dicom_id = file_path.stem
    lista_immagini.append({'dicom_id': dicom_id, 'path_immagine': str(file_path)})
immagini = pd.DataFrame(lista_immagini)
immagini['path_immagine'] = immagini['path_immagine'].str.replace(str(pos_dataset+"/"), "data/", regex=False) # pulisci il path
immagini['dicom_id'] = immagini['dicom_id'].astype(str)
print(f"{len(immagini)} immagini")
immagini.head()

In [57]:
def leggi_singolo_referto(file_path):
    """Funzione helper per la lettura parallela"""
    study_id = file_path.stem.replace("s", "")
    with open(file_path, 'r', encoding='utf-8') as f:
        testo = f.read()
    return {'study_id': str(study_id), 'testo_referto': testo}

lista_referti = []
ricerca_referti = Path(os.path.join(pos_dataset, "mimic-cxr-reports"))
files = list(ricerca_referti.rglob('*.txt'))

print(f"Inizio caricamento di {len(files)} file...")

# Usa ThreadPoolExecutor per saturare il Disk I/O
# Su Lightning AI T4 puoi usare tranquillamente 8 o 16 worker
with ThreadPoolExecutor(max_workers=8) as executor:
    lista_referti = list(tqdm(executor.map(leggi_singolo_referto, files), total=len(files)))

referti = pd.DataFrame(lista_referti)
print(f"Caricati {len(referti)} referti.")

## Prima pulizia, selezione e join dei dati

In [58]:
dataset = pazienti.merge(metadati, on=['subject_id'], how='inner')
dataset = dataset.merge(immagini, on=['dicom_id'], how='inner')
dataset = dataset.merge(referti, on=['study_id'], how='inner')
dataset = dataset.merge(visite, on=['subject_id'], how='inner')
print(len(dataset))
dataset

In [59]:
condizioni = [
    # STADIO IV: Metastasi (Il più grave)
    diagnosi['icd_code'].str.startswith(('196', '197', '198', '199', 'C77', 'C78', 'C79'), na=False),
    
    # STADIO I-III: Tumore Primario Invasivo
    diagnosi['icd_code'].str.startswith(('162', 'C34'), na=False),
    
    # Livello 2 - INJURIES (Traumi, fratture, contusioni)
    # Codici ICD-9 (800-999) e ICD-10 (S e T)
    diagnosi['icd_code'].str.startswith(('8', '9', 'S', 'T'), na=False),
    
    # A RISCHIO: Noduli o ombre (Sospetti non confermati)
    diagnosi['icd_code'].str.contains('793.1|R91', na=False)
]
valori = [4, 3, 2, 1]
diagnosi['numero_severita'] = np.select(condizioni, valori, default=0)
etichette_pazienti = diagnosi.groupby('subject_id')['numero_severita'].max().reset_index()
mappa_severita = {
    4: 'METASTATICO',
    3: 'PRIMARIO',
    2: 'TRAUMI',
    1: 'A RISCHIO',
    0: 'NEGATIVO'
}
etichette_pazienti['stato_clinico'] = etichette_pazienti['numero_severita'].map(mappa_severita)

In [60]:
dataset = dataset.merge(etichette_pazienti, on=['subject_id'], how='left')
dataset['numero_severita'] = pd.to_numeric(dataset['numero_severita'], errors='coerce')
dataset['numero_severita'] = dataset['numero_severita'].fillna(0)
dataset['numero_severita'] = dataset['numero_severita'].astype('int64')
dataset['stato_clinico'] = dataset['stato_clinico'].fillna('NEGATIVO')
dataset

# Parte 2: Gestione dei valori nulli e correlazione dei dati

In [61]:
dataset.info()

In [62]:
def plot_valori_mancanti(dataset):
    valori_nulli = dataset.isnull().sum()
    valori_nulli = valori_nulli[valori_nulli > 0]
    percentuale = valori_nulli / len(dataset) * 100
    plt.figure(figsize=(12, 7), dpi=100)
    valori_nulli.plot(kind='bar', edgecolor='black')
    for chiave, valore in enumerate(valori_nulli):
        plt.text(chiave, valore + (max(valori_nulli) * 0.01), f"{percentuale.iloc[chiave]:.2f}%", ha='center', fontsize=10)
    plt.title('Valori mancanti per ogni features')
    plt.ylabel('Numero di valori mancanti')
    plt.xlabel('Features')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
        
plot_valori_mancanti(dataset)

I valori mancanti significativi sono i seguenti:
- dod (buona parte), che rappresenta la data di morte (sicuramente rumore);
- PatientOrientationCodeSequence_CodeMeaning, che è un dato molto tecnico e spesso ridondante;
- PerformedProcedureStepDescription, anch'esso ridondante dato che ci sono i reperti testuali;
- ViewCodeSequence_CodeMeaning, ennesima ripetizione verbosa.<br>
Per questo motivo le feature si possono eliminare tranquillamente.<br>
Inoltre altri dati superfli sono:
- Rows e Columns, perché avviene comunque la ridimensione del dataset;
- anchor_year_group ed anchor_year, poiché sono solamente dati amministrativi;
- ProcedureCodeSequence_CodeMeaning (un'altra ridondanza)

In [63]:
colonne_da_eliminare = [
    'dod',
    'PatientOrientationCodeSequence_CodeMeaning',
    'PerformedProcedureStepDescription',
    'ViewCodeSequence_CodeMeaning',
    'ProcedureCodeSequence_CodeMeaning',
    'Rows',
    'Columns',
    'anchor_year_group',
    'anchor_year'
]
dataset = dataset.drop(columns=colonne_da_eliminare, errors='ignore')
print(dataset.columns.tolist())

## Normalizzazione dei dati

In [64]:
dataset.head()

In [65]:
colonne_feature = ['gender', 'anchor_age', 'ViewPosition', 'StudyDate', 'StudyTime']
feature = dataset[colonne_feature].copy()
encoder = LabelEncoder()
feature['gender'] = encoder.fit_transform(feature['gender'])
feature['ViewPosition'] = encoder.fit_transform(feature['ViewPosition'])

## Matrice di correlazione e calcolo del VIF

In [66]:
plt.figure(figsize=(12, 6))
plt.title('Matrice di correlazione per ranghi di Spearman')
sns.heatmap(feature.corr(method='spearman'), annot=True, cmap='coolwarm')
plt.show()

Come si può notare, non si hanno correlazioni dei dati significative. Quelle che sono leggermente più alte sono tra ViewPosition ed anchor_age e tra ViewPosition e StudyTime. Adesso, si calcolano il VIF, per verificare se è presente multicolinearità.

In [67]:
vif_data = pd.DataFrame()
vif_data['feature'] = colonne_feature
feature_const = sm.add_constant(feature)
vif_data['VIF'] = [variance_inflation_factor(feature_const.values, ind+1) for ind in range(feature.shape[1])]
print(vif_data)

Come si può notare, ogni feature possiede colinearità debole, perciò è del tutto inutile eseguire procedure di riduzione di dimensionalità.

# Parte 3: Distribuzione dei dati

L'unico dato rilevante per calcolare gli outliers è l'anchor_age, poiché è l'unica variabile continua e può mostrare la demografia dei pazienti. I box-plot sugli altri dati non fornirebbe dati interessanti per il modello. Perciò, il box-plot viene fatto su anchor_age diviso per stadio_clinico, per vedere le fascie di età più colpite.

In [68]:
fig, axis = plt.subplots(1, 2, figsize=(20, 10))
ordine_clinico = [
    'NEGATIVO', 
    'A RISCHIO',
    'I-III STADIO: PRIMARIO', 
    'IV STADIO: METASTATICO'
]
sns.histplot(dataset['anchor_age'], kde=True, ax=axis[0], color='steelblue')
sns.boxplot(x='stato_clinico', y='anchor_age', data=dataset, order=ordine_clinico ,ax=axis[1], hue='stato_clinico', legend=False)
axis[0].set_title("Distribuzione dell'anchor age")
axis[1].set_title("Box-plot anchor age")
plt.tight_layout()
plt.show()

Il grafico a sinistra mostra che la frequenza assoluta dell'età all'interno del dataset:
- concentrazione anagrafica: la distribuzione è asimmetrica, con una chiara pendenza verso le fasce d'età più alte;
- picco di incidenza: la moda del campione si colloca tra i 60 e i 65 anni, che è l'età media tipica in cui vengono prescritti esami radiologici per indagini polmonari e oncologiche;
- fasce giovani: è visibile una marcata coda a sinistra con un picco minore intorno ai 25-30 anni, che rappresenta i pazienti più giovani sottoposti a screening.

Il grafico a destra scompone l'età in base alle quattro categorie diagnostiche, permettendo di osservare le variazioni demografiche a seconda della severità della malattia:
- Negativo: essendo la classe maggioritaria di controllo, presenta una distribuzione estremamente ampia che copre l'intero range anagrafico (dai 18 ai 90 anni), con una mediana che si attesta intorno ai 61 anni;
- A Rischio: presenta una mediana più alta e una variabilità estesa;
- Stadio 0: non presenti nel dataset;
- I-III Stadio: questa classe registra la mediana più elevata del dataset, che conferma il fatto che statisticamente i tumori primari vengono diagnosticati prevalentemente nella popolazione geriatrica;
- IV Stadio: mostra molti outlier inferiori marcati, indicando un sottogruppo di pazienti eccezionalmente giovani (tra i 20 e i 30 anni) colpiti da malattia in stadio avanzato.

In [69]:
fig, axis = plt.subplots(1, 2, figsize=(20, 10))
target_counts = dataset['stato_clinico'].value_counts().reindex(ordine_clinico)
sns.barplot(x=target_counts.values, y=ordine_clinico, ax=axis[0])
axis[1].pie(target_counts.values, labels=None, autopct='%1.1f%%', startangle=90)
axis[0].set_title("Distribuzione del target")
axis[1].set_title("Percentuale distribuzione del target")
axis[1].legend(labels=ordine_clinico, loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

Come si può notare, il dataset è molto sbilanciato di casi negativi. Per il momento, escludiamo i negativi.

In [70]:
fig, axis = plt.subplots(1, 2, figsize=(20, 10))
tipi_positivi = [
    'A RISCHIO', 
    'I-III STADIO: PRIMARIO', 
    'IV STADIO: METASTATICO'
]
positivi = dataset.query("stato_clinico != 'NEGATIVO'")
conta_positivi = positivi['stato_clinico'].value_counts().reindex(tipi_positivi)
sns.barplot(x=conta_positivi.values, y=conta_positivi.index, ax=axis[0])
axis[1].pie(conta_positivi.values, labels=None, autopct='%1.1f%%', startangle=90)
axis[0].set_title("Distribuzione dei target positivi")
axis[1].set_title("Percentuale distribuzione dei target positivi")
axis[1].legend(labels=tipi_positivi, loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

Come si può notare, il dataset dei positivi risulta leggermente più bilanciato. Perciò, si procede con il data augumentation del I-III stadio, poiché i casi sono veramente rari; e con l'Weighted Cross-Entropy Loss, perché non è presente molta differenza con quelli a rischio. 

In [71]:
dataset['numero_severita'] = dataset['numero_severita'].replace({3: 2, 4: 3})
mappa_nomi = {
    0: "NEGATIVO",
    1: "A RISCHIO",
    2: "PRIMARIO",
    3: "METASTATICO"
}
dataset['stato_clinico'] = dataset['numero_severita'].map(mappa_nomi)

In [72]:
mapping_gender = {'M': 0, 'F': 1}
dataset['gender'] = dataset['gender'].map(mapping_gender)
posizioni_uniche = dataset['ViewPosition'].unique()
mapping_view = {pos: ind for ind, pos in enumerate(posizioni_uniche)}
dataset['ViewPosition'] = dataset['ViewPosition'].map(mapping_view)
dataset['StudyDate'] = pd.to_numeric(dataset['StudyDate'], errors='coerce').fillna(0)
dataset['StudyTime'] = pd.to_numeric(dataset['StudyTime'], errors='coerce').fillna(0)
print("Dtypes dopo la conversione:")
print(dataset[['gender', 'ViewPosition', 'StudyDate', 'StudyTime']].dtypes)

In [73]:
dataset_pazienti = dataset[['subject_id', 'numero_severita']].drop_duplicates(subset=['subject_id'])
train_pazienti, temp_pazienti = train_test_split(
    dataset_pazienti,
    test_size = 0.20,
    stratify = dataset_pazienti['numero_severita'],
    random_state = 42
)
val_pazienti, test_pazienti = train_test_split(
    temp_pazienti,
    test_size = 0.50,
    stratify = temp_pazienti['numero_severita'],
    random_state = 42
)
train_set = dataset[dataset['subject_id'].isin(train_pazienti['subject_id'])].copy()
validation_set = dataset[dataset['subject_id'].isin(val_pazienti['subject_id'])].copy()
test_set = dataset[dataset['subject_id'].isin(test_pazienti['subject_id'])].copy()
train_set

## Data augmentation

In [74]:
base = T.Compose([
    T.Resize((224, 224)), # Risoluzione tipica per le CNN
    T.ToTensor(),
    T.Normalize(mean=[0.5], std=[0.5]) # Uso immagini in scala di grigi
])

augmentation = T.Compose([
    T.Resize((224, 224)),
    T.RandomRotation(degrees=5), # massimo 5°
    T.RandomAffine(degrees=0, translate=(0.02, 0.02)), # piccole traslazioni
    T.ToTensor(),
    T.Normalize(mean=[0.5], std=[0.5])
])

## Undersampling

Per eseguire l'undersampling, trovo i pazienti sani simili, con la logica degli N-grammi e la Cosine Similarity.

In [75]:
train_negativi = train_set[train_set['numero_severita'] == 0].reset_index(drop=True)
train_positivi = train_set[train_set['numero_severita'] != 0].reset_index(drop=True)
vectorizer = CountVectorizer(ngram_range=(1, 2), binary=True)
testo_negativi = train_negativi['testo_referto'].fillna('')
ngram_matrix = vectorizer.fit_transform(testo_negativi)
sim_matrix = cosine_similarity(ngram_matrix, dense_output=False)

soglia = 0.8
pazienti_buoni = []
scartati = np.zeros(sim_matrix.shape[0], dtype=bool)

for ind in range(sim_matrix.shape[0]):
    if scartati[ind]:
        continue
    pazienti_buoni.append(ind)
    record = sim_matrix.getrow(ind).toarray().flatten()
    cloni = np.where(record > soglia)[0]

    for clone in cloni:
        if clone != ind:
            scartati[clone] = True

target_negativi = 3000
filtrati = train_negativi.iloc[pazienti_buoni]
filtrati = filtrati.sample(n=target_negativi, random_state=42)
train_set = pd.concat([filtrati, train_positivi]).sample(frac=1, random_state=42).reset_index(drop=True)

In [76]:
# Verifichiamo la distribuzione finale nel set di addestramento
print("--- DISTRIBUZIONE FINALE TRAIN SET ---")
print(train_set['stato_clinico'].value_counts())
print(f"\nTotale immagini per l'addestramento: {len(train_set)}")

## Weighted Cross-Entropy

In [77]:
def calcola_pesi_loss(dataframe, augmentation=True):
    conta_tipi_target = dataframe['numero_severita'].value_counts()
    num_negativo = conta_tipi_target.get(0, 1)
    num_rischio = conta_tipi_target.get(1, 1)
    num_primario = conta_tipi_target.get(2, 1)
    num_metastatico = conta_tipi_target.get(3, 1)

    if augmentation:
        pesi = [1.0, num_negativo/num_rischio, 1.0, num_negativo/num_metastatico]
    else:
        pesi = [1.0, num_negativo/num_rischio, num_negativo/num_primario, num_negativo/num_metastatico]
    print(pesi)
    return torch.tensor(pesi, dtype=torch.float32)

Con Augmentation

In [78]:
pesi_aug = calcola_pesi_loss(train_set, augmentation=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterio = nn.CrossEntropyLoss(weight=pesi_aug.to(device))
train_set_aug = RXToraceDataset(
    dataframe=train_set,
    cartella=ricerca,
    transform_base=base,
    transform_aug=augmentation 
)

Senza Augmentation

In [79]:
pesi_no_aug = calcola_pesi_loss(train_set, augmentation=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterio = nn.CrossEntropyLoss(weight=pesi_aug.to(device))
train_set_aug = RXToraceDataset(
    dataframe=train_set,
    cartella=ricerca,
    transform_base=base,
    transform_aug=None 
)

In [80]:
dataset

# Parte 4: Addestramento e risultati

In [81]:
ricerca_train = os.path.join(os.getcwd())

In [82]:
# Training clinico
feat_cols = ['gender', 'anchor_age', 'ViewPosition', 'StudyDate', 'StudyTime']
clinical_model = train_modello_clinico(train_set, validation_set, feat_cols)

In [ ]:
from PIL import Image

# Disabilita il limite di sicurezza per le immagini mediche ad altissima risoluzione
Image.MAX_IMAGE_PIXELS = None

In [83]:
# Training modello visivo con augmentation
model_aug = train_modello_visivo(train_set, validation_set, ricerca_train, base, augmentation, 
                               pesi_aug, device, use_aug=True)

In [ ]:
# Training modello visivo senza augmentation
model_noaug = train_modello_visivo(train_set, validation_set, ricerca_train, base, augmentation, 
                                 pesi_no_aug, device, use_aug=False)

In [ ]:
dataset_test = RXToraceDataset(dataframe=test_set, cartella=ricerca_train, transform_base=base, transform_aug=None)

In [ ]:
def estrai_predizioni_test(v_model, c_model, test_df, dataset_test, mappa_nomi, device):
    """
    Esegue l'inferenza una sola volta su tutto il test set e salva i risultati grezzi.
    """
    v_model.eval()
    fusion_sys = DiagnosiIntegrata(v_model, c_model, device=str(device))
    results = []
    
    totale = len(test_df)
    print(f"Estrazione predizioni in corso su {totale} pazienti...")
    
    for idx in range(totale):
        if idx > 0 and idx % 50 == 0:
            print(f"Elaborati {idx}/{totale} pazienti...")

        row = test_df.iloc[[idx]]
        img_tensor, label_tensor, _ = dataset_test[idx]
        label_reale = label_tensor.item()
        nome_classe_reale = mappa_nomi[label_reale]
        
        # Diagnosi Integrata
        output_integrato = fusion_sys.diagnosi(img_tensor, row)
        predizione_finale = output_integrato['diagnosi']
        
        # Confidenze Pure
        with torch.no_grad():
            img_device = img_tensor.to(device).unsqueeze(0)
            logits, _ = v_model(img_device)
            prob_vision_dist = torch.softmax(logits, dim=1).cpu().numpy()[0]
            prob_vision = prob_vision_dist[label_reale]
        
        prob_clinical_dist = c_model.predict(row)
        prob_clinical = prob_clinical_dist[label_reale]
        
        results.append({
            'classe_reale': nome_classe_reale,
            'predizione_integrata': predizione_finale, # FONDAMENTALE PER LA MATRICE
            'conf_vision': prob_vision,
            'conf_clinical': prob_clinical,
            'corretta': predizione_finale == nome_classe_reale
        })

    print("Estrazione completata!")
    return pd.DataFrame(results)

In [ ]:
def plot_scatterplot_fusione(res_df):
    """
    Genera lo scatterplot delle confidenze Visione vs Clinica.
    """
    plt.figure(figsize=(12, 8))
    palette = {"NEGATIVO": "#2ecc71", "A RISCHIO": "#f1c40f", "PRIMARIO": "#e67e22", "METASTATICO": "#e74c3c"}
    
    sns.scatterplot(
        data=res_df, 
        x='conf_vision', 
        y='conf_clinical', 
        hue='classe_reale', 
        style='corretta',
        markers={True: "o", False: "X"},
        palette=palette,
        s=100, 
        alpha=0.7
    )
    
    plt.axline((0, 0), slope=1, color='black', linestyle='--', alpha=0.4, label='Equilibrio Visione/Clinica')
    plt.title("Analisi della Fusione: Confidenza Visione vs Clinica", fontsize=14)
    plt.xlabel("Confidenza Pura Ramo Visivo", fontsize=12)
    plt.ylabel("Confidenza Pura Ramo Clinico", fontsize=12)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title="Legenda")
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.savefig('analisi_fusione_scatter.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix

def plot_matrice_confusione(res_df, lista_classi):
    """
    Genera la heatmap della matrice di confusione del sistema integrato.
    """
    # Estraiamo le colonne che ci interessano dal DataFrame
    y_reali = res_df['classe_reale']
    y_predetti = res_df['predizione_integrata']
    
    # Calcolo matematico
    cm = confusion_matrix(y_reali, y_predetti, labels=lista_classi)
    
    # Disegno grafico
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=lista_classi, yticklabels=lista_classi,
                cbar_kws={'label': 'Numero di Pazienti'})
    
    plt.title('Matrice di Confusione - Diagnosi Integrata (Visiva + Clinica)', fontsize=14, pad=20)
    plt.xlabel('Diagnosi PREVISTA dal Sistema', fontsize=12, labelpad=10)
    plt.ylabel('Diagnosi REALE (Ground Truth)', fontsize=12, labelpad=10)
    
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig('matrice_confusione_integrata.png', dpi=300)
    plt.show()

In [ ]:
# 1. Estrai i dati (Ci metterà qualche minuto, usa la GPU)
df_risultati = estrai_predizioni_test(model_aug, clinical_model, test_set, dataset_test, mappa_nomi, device)
classi_ordinate = ["NEGATIVO", "A RISCHIO", "PRIMARIO", "METASTATICO"]

In [ ]:
df_noaug = estrai_predizioni_test(model_noaug, clinical_model, test_set, dataset_test, mappa_nomi, device)

In [ ]:
plot_matrice_confusione(df_risultati, classi_ordinate)

In [ ]:
plot_matrice_confusione(df_noaug, classi_ordinate)

In [ ]:
plot_scatterplot_fusione(df_risultati)

In [ ]:
plot_scatterplot_fusione(df_noaug)